# Full-res preprocessing CACHE job (one-time)

Streams the REAL full-res data once and writes the compacted cache that Phase B
(U-Net) and Phase C (world model) both read — **one file per (variable, scenario)**
(`X_<var>_ssp126.zarr`, `X_<var>_ssp370.zarr`, …), uploaded as per-scenario private
datasets (`climx-cache-ssp126-1`, `climx-cache-ssp370-1`, …). Scenario-pure files
are what the world model needs (contiguous daily sequences within a scenario).

**Disk strategy** (Kaggle has a 20 GB /kaggle/working but ~1 TB /tmp): zarr chunks
are written to `/tmp/full_cache` and **auto-uploaded in ~20 GB batches as private
Kaggle datasets**, then deleted locally. A manifest in
`/kaggle/working/data/full_cache_meta/` (persistent) tracks progress, so a 9 h
session kill loses only the batch in flight.

**Accelerator: NONE (CPU session)** — dask streaming + writes; doesn't touch GPU quota.

**RESUMABLE**: re-open and re-run until the output ends in `CACHE_OK`. Then commit the
printed manifest into the repo (`docs/full_cache_manifest.json`) — the training
notebook uses it to know which datasets to attach.

**Sizes**: `--subsample 3` shrinks the cache to ~1/3 (Phase B only). Default ×1 is
contiguous daily — Phase C-ready.

import os, subprocess, sys, pathlib
repo = "/kaggle/working/climx"
if os.path.isdir(repo):
    r = subprocess.run(["git", "-C", repo, "pull", "--ff-only"], capture_output=True, text=True)
    print(r.stdout or r.stderr)
else:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/nonewanit/climx-emulator.git", repo], check=True)
os.chdir(repo)
sys.path.insert(0, str(pathlib.Path(repo) / "src"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
r = subprocess.run(["git", "-C", repo, "log", "-1", "--oneline"], capture_output=True, text=True)
print("HEAD:", r.stdout.strip(), "— MUST be the commit None told you, else the pull missed the fix")
print("setup done")

In [ ]:
import os, subprocess, sys, pathlib
repo = "/kaggle/working/climx"
if os.path.isdir(repo):
    r = subprocess.run(["git", "-C", repo, "pull", "--ff-only"], capture_output=True, text=True)
    print(r.stdout or r.stderr)
else:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/nonewanit/climx-emulator.git", repo], check=True)
os.chdir(repo)
sys.path.insert(0, str(pathlib.Path(repo) / "src"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
print("setup done")

## 2. Run the cache job (live bars + ETA; re-run after any kill until CACHE_OK)

In [ ]:
%run scripts/full_cache.py --subsample 1

## 3. After CACHE_OK: commit the manifest

The last log lines print the dataset slugs and the manifest JSON. Paste the manifest
into the repo as `docs/full_cache_manifest.json` (or send it to None) and add your
teammates as collaborators on each private dataset — that's the whole setup.